# M4 v2 — Complete 3-seed manufacturing knowledge revision study (accelerated)
**Self-contained**: original graph and scripts embedded; no ZIPs or prior notebooks needed. **The complete final study has 3 training seeds, 4 model roles O/R/U/C, and 3 evaluation representations BF16, INT8, NF4, on all 40/240/80 test facts.**

**Start here:** Runtime → Change runtime type → GPU (T4 or better); then Runtime → Run all. Authorize Google Drive. If disconnected, reconnect and rerun **Run all**: finished training and evaluations are skipped when their frozen configurations match, partial evaluations resume. The original v1 output directory is not modified.

**Research integrity:** v2 changes **O/R supervised training exposure** relative to pilot v1 (6 old-record exposures, 5 epochs, batch 8). O and R have matched supervised step budgets. Development criteria remain **20% original recovery / 10pp separation**; results are recorded even when the gate fails. This is intentional: continue a complete *negative* study, never relabel failure as successful unlearning. The final-test configuration freezes automatically *before* any final-test prediction. A model with an invalid original/reference gate cannot support a successful-unlearning-durability claim.

**Timing:** no code can guarantee completion within 4 hours on a shared Colab T4, especially if the runtime disconnects. This version uses batched answer NLL, partial-write evaluation resumes, local weight caching, and no unnecessary development re-runs. Do not shorten final test or remove seeds for the paper.

## 1 — Restore the exact published Chinese graph and current M4 code (CPU)

In [ ]:
import base64, zlib, hashlib, json, os, sys, subprocess, time
from pathlib import Path
WORK=Path('/content/m4_v2') if Path('/content').exists() else Path.cwd()/'m4_v2'
WORK.mkdir(parents=True,exist_ok=True)
PAYLOAD={'mpkg_v1.json': '...', 'm4_build_dataset.py': 'eNqtOvtzG9W5v/uv2NC5Pbs3K1mSHyFytzNp4raZgpPG4bYzRrOz2j2SFq92N/uw7Gg0k1IgSSHQEpheAhdKb9IwdAihnSnESbh/zLVk8xP/Qr/vPPYhy6EFApNI55zveb730Q+OzadxNN92/XnqbynhTtIL/IW5p5566tlFJQ7SyKaVbhSkvkMdpW/5aceykzRy/a7yy1PKwE16Ct0OPdd2E29HiXf8pEcT12Y7TmQNfCWiW27sBn5cnVsLFM/yu6nVpZV+4FBPSWOquL7iWIml2HAmiVI7gcMriuV5kr5DI3cLyFt+PKBRDGyEShIoQeR2Xd/ylCgYAPIzNHa7vtKJgsvUV9q0E0RU4VSCNLGDPlKKQ8rwV5XzTFJloVqvHdeVOHE8t60EvrdTReHnAE1fMc1OCsJS01TcfhhECbDgB4mFGOI5sdSz4h7A6soLceDrSmT5TtCHf6kOEnkepxfrihV1QyuKKUcdWglCSbzn4evc3Prq6hmjUWss1+u1+tzqr8+vnr64esZcP/fchdOr5vrPTxmkbncWaidPLiwttTv1E3ZnaXnhxOLi0snaCWotthdOOAvLnU57eaFWP9FYtpdP1J1O267ZbdqwaO3kUofM/fzcM2fOPXfR/NW5C784f3b19Oq6oZLJ2/+7/9tP9q9f3X/4kvKT1UrjFNHLiz9r1KaWzi5VavX61OLTlYXiirJe+dnSAtHm1sxzP1k/98zqxVVjsbairJkXVi+eOrtmNPi359ZA9jXj6drchdVnTl08e24NuYqoZ4ZRYNM4pg6gxe+D0AQ1Wn2agClkazGYCM2/JjshFV9CO8NgtneyxcNIigc7cHHBANiec2hHsS0/8NWEbidaM6JgEGjU1ThtqxF5Pj5OdIUQHUyXH9Gq8NENVaI8ny7UarX/v/Jitfn1o1uADZHFPStov6BuZaiE/VRho7G0rKIVVZ20H8bqlk79GM3Pim3XNX5qeTHVYzAXc5PuxMbFKIWvFAVJgigGfenwX5MAB9S3wfBVkiadytO40KPbjtulcaJyLjquo4LQupuxQdYvnK6Q44K9IaqE2Tlp4jkCPkaa7kjbaNYbLa6WvrVJTQwH6qbrOzqc0OFfuq015xT4A/p5QTd1wGbA1kZzobUCwgP3BiHsgNtRENAwpu6Zg+OfS6nlMeBFhPWCxMCVFWUTZI0TGoLWkQguajmQsdEhkz89mNy4N772j6+u/Gn8+Z2D67vjd788uP/Z149eGyLMaP/WS0OEG03++9O9h1f2Hlwb/w5P4v+7b3z96AOidwg/On7vI350fO/65PU393ZfEQAM7+STO199fFsATD64c/Dpn4doCwiqMTL4jbE4AisYf/pgvPsWLBfJjX9/Y//u/YwTRNZi8lCvrKSy8eeKkkpGHTOlAe/4vdECmteGXIXw8TpZydWIWhwE0WboUpvmeL8XpV6/cnDv/vjx2yjoGy9O3r4/vvKoJN8M7TLNjG9+vPfFxxyGg4nTB59+Pnnj93tfvstp7n3xyZFqRqKvfzC++yrnAXCADx6pURE6pqyOiOUpfck4w8j+m9oZf/b+3u7rXMC93bvjm7/Z//vD8bU/ltQBvPPV8eM3AWBaZ9zAOHRZfgF17ZX9zx5ye3qS0CxATovMFqcE5pH0O4m7f/cGHNh7+A/k7P1XS+Ie/PmvKOjLdzKXEueu/fUbJS4iPVrWqfg/LTO9lLphn/rJtOD5xreTnjn23uP/23/ro2J8mby3O979nZTt+t2DD2cAcKXxowf3vhzfvlpSBidYUEYWVDjIN11/OfvNjCJZ5G20pvQitPm9xAsucxY1uNdzp50dLCY3b+w9fg99nwEApDgtA8z49q3x/T88WVvlYMEjxRNjxIzKYHaOmqEpB/L396chLv/ktat7D2/P1lB2lOnpFs9O45c/33t8c/zKy+N7D2ZHlENKkvF1KqIKBhjWosJi2owsF4r5X9Cd1SgKIlYTcGlFgTEkLuR2LDxYucBLBZ3wGt/Myw22my0HA5OdNGttixUH7JueqRH/EGQZ6mzSZHomeA3wGf7WCe8Z4BvIAvYNBOFa+mECdn9JJ+DebrIj4Mo4e9RyTDc2swzJOcNu5bsWpto0+7msvAgEhngZBktaURdQjEGVRaD8A25GvAxjzRlsQgsHAS5TpBoGgafbsJno8aYbmq4TGzGFAlDUZ+3U3qRJbAw3m1hTUkdVtxVompRtlBGhsULb3jh0Qy3D2IRGyME9uNKWAi0Rgkgqmg78GZ7VbzuWcrl5mZ/SGO5NPJiV+CPGyA8UB3XWd303xtaxbUGTCJ4DDR52ftBpQk8J6Cn2UtC4RS50fol7mSrUD9IuNKDsFivY27kdQBBRG0ChJWRBLU2MDW6mg57rUcWjvgqL2o+YbnJHhvvsRuxCeaWdbRzmu1m6P9CS0OXGZquJBKtWGFLfUfPlahiEak3TVgpUsH6fRiSZMwzOXTui1uZc4QDqOschnO6/LC+l3O065Bz0sMpQIhrNDxmmkWJtWa5ntT2wv6JbwiFuR+3U9RyVX3YT+1EdttgHWdBHtsG3qxE6R3snobGqrSi8szCmGhk4Xu47hAT8+zFjRns7S5x1RhHbJqXvxn0rsXvKkOMYKccMZTgDz0iI6Bism/ICy4kZPysK9iuxNAi8WTDrI+4Wd11sa3Cf+mmfQqNFVWcj741aG/CxpR2yB1g9Oncw31HxgniC1X68sAy2Dss7aoy0+DKjH/N4M/78L5AYIIqM3/i7+PDul/u3d/HDX17kK/XKEvzdYP13o7IIXZ/WxJlK4vpTdsZ0IG007+Kwz2NNnKYJvzwDfmknipPigAdkdyDbgd5xnjGP4TVuKnTb9lKHsnENeLzbTYM0Bl/tWz76MjvFHZF9NAojkeppNEwKbTPgp2png8U1ESg6KDhjlDMj7q0ztYnKZog3ppBAkKrzS8a4N+yIUFWGHq1YcDFRwtwOI5dh4KcCWRx9hUfQBZyHk0SL3S5syZzUQpjDM5fWXMHjORXtR3IO8mS3FqdHSo96TgWcVMnIc+aE9eM2Rr+jEgTHo0uiJSBzttbErozbp+DSuTrAzmwa+RisxZmcqVgaiaOw0VcC0rGTKt2CuJ70ApBV5JCnawrdArEZh1pVlmNdF+LW7FtAQPU7XYXWEuIs1qAEjgMPrD6fWipBBwIupduYnIDpDH9FOLZMN7oShxgXFcuOAljO5edi2MCN64AXlc1JSicsakaynao2OLdwkYOw5E0Qw63USxwXXNkDZrUsxLGEnpNvMtiN7YJiWjIabOdQLOWxo03+D6YxrBTUmemdJ1sP0gLUafGMlCu30MzlLDAPnHbP8rszU+8gZMUFL1EYI4fjLedvELaaGZksB8utLAVLUkfm34xTwyiwOjMVC1yHHZac9eO0A9WIC9UJ5Dw0HpulbrxGNCQWtkhZb0e4XcYRd7wA6qHoSLeWBqXLGWslF0IvEsrKAHBHIyNxnGEvRicsGg+xxME07ZghyczQwQV2KM8fRCaW9eydAOhKpSg+FM+xqPNksMDXBsvLjmwFttXWoVZJZMSArKngLBD9NAYEud9xt2MQhjAfFAXsXjQFrdw7snCy/a+4oEiQ+fCMc3mkOwLZ3K/y8NUsWtO/5PvNaZIbnbIXOw5elZQvNy4Ro4qFT9mycl4sD3KyDxxsoRtvsbNbeJZpElndkrp/MjetaWcpYp5hLWsBXGCHFh6RigBEK3JoFLc2XB90zDsmYag4p6619Pqy9h9owsXTWs4XBHmDrJ99tiL9o0KOCwQbi838HMpkFGXjnT6uYKvZKjf14/c+Onjn1uT9l9iM6m+TN29/9daVyR/v7l8XI+PxjQ+HQBp77NLoFJGN+EBj74tdNii88tVvbhZG0sVwJVv3yUcfTl79gOPnwAz5149uTWGGDn7yP3fGr7zDZ2d87lweB5Tx731x4+DKbw/e+cP+6/cPPnyNMw8c4yj2tavjq7u5dIwHYICRzka/nIG9B68KBhjRfGwwZZwyYvNBAYE8nV9H1gSj45hskCA3ZKZmq/BlqrmWAwAwAZ3YaRRBPDblYsFXyoOBQ1MJklmlKWsE6ZdkiiIUIbHbTz0smEmTvdIcPaEYyZC4us0eAkMaseAAfaRP81IJnJBhdPGJEXfQKx2on7yADSl1VuvgRkwtD4MnvoxWINsnRdiqLOwL+QhlVtjBsBdZLDUBWT/u0EhnbovsVDykpeDwBe+I+jK8Qp3vdrD7GxJ8xIVIB7p6drFSeiqubPrBALjq0oq8LWhS8Ly5RaOYK9gOGxX59pmpVF6EE7hwpF6r1k8sLzTmnU64dXJ5wV8cVPEBUtoG1BrZBIW3iLBH8RbwZTXDKd+gAePlXuX0mnhn1hX+rTyNkFYBmBIrSWM0BVfqszmrYLSiQgzTRaYsPF+z/fLLei5vL+D1d149kiZWc+qMulUnftDHezZZiw8nh0TyQ5qFrE8kD7jKs7VOZKUvbTzuuWGMB3g7MMpYwhEgzlX4KdLcIM/5oFWsAXJI0Skx4XAeA//MaARYPZc1Kxk8PgwCYF4JZPaCZSsW2GlUhRu4iO4wD2Y/z+xVNqOI0gEz4/iZd4DlMVZ4opJGm4jWF1BBsqH90I1ALI+lnWRnPgixwweEsFT+qQO0JFg7owdxWHxehbYGGGMMQO3BfmkQ42uuEtgszrBZYdJzY/7zBmgO2FMtQ9LKtCuVIO7QzOt0tGEoHmZ1zLOKhem2WTuSBLeGb41eln5H4k+ZeXxr/MJuNG0kB3jV/qbjRiqoD+5QPHyzhswMNtk3Me6hHfYjDE/FSIDVpC5SS6FjYD9ZwdHYvDykVeHifZUMiM7ezeHCDflyrsB1DsrtBvJ5meuB4W4OqoPITWjx5f7yN7/ca8fJ874oajjTmZea3ErZKtGFuosHpeuWzkm1FQ/OSFolmEL2zQtUUHPiskyDn1DSDclbxo2u5pGmhAU2xPVnDLXKI7WYW4PubgvULMkRvYaw4NtEr+Mn9HCiN4rQopTM+IPyWNBi2U8gPmZw2ObM2RdcVB/K2iHJColtWUjIcMKWZNhvbbjbrfwZodA8lLP+9DA/R1/tQtMxvawLqvgeIcUhzVzzs3GXK4uiHrLLGBVG+LDfKiESzkGGGejIHAq1jaRNMA1ppeSOE0+8XhNHvBCYWsYwrKLrNKdmvmFpMlwa/3LTKrTy6NddL2ir5D8FaenxzDuJJM42icadzMSf2BQ9TZ6a5XAui/lGg7vaYecWxRD+XMdGO9+yIteCAMPWC5NBYfNG1uoyc5OjAmn5xWFBdkD6AG7yvFpEjtnpSd31DwvTuCKc5Xlqaa4l+rHDNcI0yqPnlto0gSA39WMGfJmqnTm3gWwiS1GkgEUOEIoupYE6FnI75ZwdF7IeP4QNrxxc9hKDsJ4Mwc5cYnOBAsHW9I0ylGLai9zx75quELptZWUxHLU2sU6c4+9D2GYWTG9Izp9aX5cVfpa3eYSFqi2bJOe1logCrFYy8YmNHxMTXV24GQDnnjdV4rXyYs7E98jS2SMSfd49lYvj0ROdpvRKJGnMzYEBmSb6vmlC3DHNPlAwTfEgboWG/Jlj9VTUTbE1OY/fIlVbsUKcT5iWWFdJpcLZIroYlhikH252za268PlZIKDGME0KIM8umqfPN8wzUGMhRNSNDQBjPCBgLF6e+PMWPmepuFrlpDU9X+GYQe5/AsQENN4=', 'm4_runner.py': 'eNrVfdtu3MiS4Lu+gu6DQZISi7rY7WOUhg2oJbnHM7Kkke0+C9QUCKrIKtFikWxedLEgYLHP8zPzvk/7K/sB+wsbl0wyealSufv0DMaNtqvIzMjIyLhnZNZfXu1WRb57HSW7YXJnZI/lTZq83vrhhx8+vjGKpR/HxixNyjyN4zAwln5Szf1ZWeVRsjBuk/Qeni5CIw/voiJKEyN8yMI8WoZJ6Wwdp0toHxRjo8z9KLGN8M6PbWPhl6ENPbI0Lx3jqkoKI0qMeR4WN8YlDW9keToLi8IAWMYyDcLY2boKf6siaGMcfzk5MgCskUVJAih9zv2kmKf5MsyL3cvT959hLmUBDa4fS2gOk4ryNEGMDGjF4GBKjJqzdQQTZFwA1jxKAphYYQQwiTt8kKdLA+frxzg1bpXlYRDNSphuYRtJeAdILsIkzH18CVPxgzs/mYUOEnErWmIvw88XmZ8XMPEZUlL1nhV3tnHjFzdxdG0bX4sUqLT0yxvbSOEtzCxIl7ZRPMKXEqgKH2+qMoq3CK8MGkI/Qw5xCV+3ti4/XJ6efTg/9X49vfr04eLcFR/feHcHnj+bhTHj6B3sHbzd39/bF1sfL05Oz7wPJ6741/sw2cW/DpwfR3vOjz+PPiRFmVezUrW6Ov31A0FMC0dS1VmEpYkDtFsIaysI50xqD1jDu703rXEeAtskxpNQzCLG7W7PRjQ32o+MMC5C4+l56/Lq4p9Pjz/Do0+uKX7zgEO+Clvcqg936kOqPiCbqc9Vpj4F6X3Cn62tk9P3R1/OPnufTk9PAKqiii0/HKgPr62ts4urI+/K3X97aNDHo7PLfzpyXx8cGh+P/of3+eJfTgGvH/cPtmjieegHHi5mbOIaWeMtA/7I+U/whROnflCYhUU8SQKAy8fNHepfhg+lGSazFDnSFVU5H70TllNkcVTGURIWpoX0KhxYpSgzrSmP7ZfpMprR6ATNTq+/SgTwq9sMc0gPHGBLlNblbRDlJn8p3M95FdrhQ1SUXnpL3ywCUS4zl3rdR+WNV1TzefRA0Bz+vCMcaCLqxs59HsEy0Fxo3kG1zAoTcLLDpKjy0POLWRS5731YZxukD0Z3D6wd8W+JsHuTr6GCwMb+LOR58LyLEBgbZRE0CPQx8btdpvmsJj8Jk4PP6aV1SG8dVGl+7PWez6rA1196oAq5gRqwBLZiHgcphOXgcaQ0EhB6QLJaalpKNTmqyvQj9n+f5sd+Vfjx2UcbH35Ob8Mk+hbm9s+gyo6S4GdUZcdpMo8WDcgsnJcK1Fma+/zeNj77xe3nxwyUBUinh60YS9u4hM80ICM6N5JUIsqzjQrPv/Oj2L+OQxRYPwLhAwWNquc0z9PcFKR9f7n8YkQFMDSp5IB4mFT8Lmp4xzi+8RMwCscpQFL9jdFP2M+RqyiFgQa3fxchbG3O9ZQ7M24mTEuGb2c3fulFIHtlemv/VoUFamIbFuc+zOUKVkWYuxPQU2kcirHAr6A50AoCd4qx6vQ8ZanKQ2B8F8A5fpbFjzxCGS6BRcvQxN7AhzwPlis/CDxpMAAMaqNlVmpCNq/ieC28nQY5vyhASH3AS8OQZ/M83WhcEj1LcQSOPRnHISgPmpc1feXyJ6YNS1KfL+qVMmqMJGWMZQRuRDm7OTSKMs2gd3kDzUrgETB1xS06EowwaMFZmudgHONHUWOEuCBW1k+Nql2Ly1ycPvhArRAs9gzktdB19FPz+XlsPKm1nIz39/amzwOj/qOrEWNQJM5Tbc5yJkT4os3rTyJKsoqYT4wRtm0IEBCwcGI8GeH429pQO7QS+tjTZ+bipX8betdIUeJh0pIdXQcTQP7J0IwgJjAmCux5moTjF5YPnBFQC4beVU4DcFmAAVn6Dyai9TDR5jNlQ/aAhkwixJ18dzI9XOJfMfzF7A0NwSgstbbNesK4Lg80wkGwXXucuqWP4hEmQb/NzqQ7+ek2fGu6LlXXyT7TvD/MzmSv0ylujydXDgbjpavbqtVOxqxZQSQLIPGdHYDbMwtdgapW2EEJ+srlJnGaLJiAiX0HZJFjTkwNJdu3bFP4JUo4Si+KjrCX+FCiYseWYpE4LQrQgzkYrt9MVoXEMG0m0QyVkyTOvErILQVXF3jgvZxMUcWlSyDMGhmXgLUIZrcxUy06+E5tUF/ezJ/dhLreidMFqHeXB3P422Rsj0f79njqzMFXKk1p/MGPDksFXa0BtN0fTyWoonDfO7McKRBi3JI9mgzRwQjDz0ITwMon9B0WcGrZDFlrY9nRIknBQ0Gv5MHFVbbB2FVEI1ckIEvCqtvL7vSFUQVLGAWufPGK+luH4N+49AL8paUZREt333JmMagrcxkl8EVnIZyL1g4YKJEaIPQTD9+uXNvCnrkvMIE+Eg9j7c7435aeKUKzts+tMXQOvqZF2n/Lmkd3KK7n+2/BUcw4dgKflTx6biM7ETRC0K1HctDFARMVkl8BHVWkwih4PDj9DYt5782yyluGSw+ch4U0d9vb7QDEasZxytRkObQO+cGMHAmnw546jZaNHxGEZZjDeoHOB1cbvCBY0W/gIebpfWHCehc2WufG7Xe1gABfO4Ff+rsQB5UwN49m6NBbqWnTOFjRJb0uADJ61L1O0mgBUOuVe/BmzwB1gg8AGDx4szdouz4kxI3AarNbcORwEKOAuRQNUJyK64oLoQI4+GcHgG4TWjVC4JKHftnpdCUazf4X45MPbmBSLa/BzKRzsM5kpYvdogyzAnXOxaHBYAAVJg59AKcgwEg7NdRodV+ncQQgTpAu/hX9w2RDh33nr39t1Lg2iQn0cWY3KWhlIhtpYA/NUo7+q6moNzhTa9qZ67FOoK3GN/nVjytJbuIKZiFaP0JRsknHE+64v953ebtuPzZhrdCJkWoK6SqBe6VV6fK0q3JXgEDMbrM0gtBQwHLlTU9rlz7izLi3n9+C91wD2RWfr44+nJ+eEK8KRTNu5lCEWUdO1N/VwmPZak043KwrJp5mwDiuco1rvGyBiMoHNFsRZunsppCP+Ist4tDPEwDuYZJEvotzux6h+SNIj3ooKLIdK1ZRK5zB/IYtOjwkO3ee2iKLshBDfA+WigF1szowJ/8uBI0HxqWeWvNkEGkUbjDDSTQHn9cDS3Xw41sxlsknh7+burJRbXnpZFqCsmqgTJ2b8CGIFvDatJ7r4VCCbsNHG7QKuXdyURz0mVrrrPiAMkjQA1QU9BmvcOdR0UKYQEKDHxp+xOTkPI5mZWFgSoJiUsAJ9MYTQEUXvxni2XjlGk8wzPOhMctRzRi+kYT3WtLSCCKMP9JcBSAc4cFApvj0Lx8uLz+c/4LKZ1aOKKbBRGKGaxew3IBHRjxrHbIukKJ967bEebVxw9jTm/tFuc6ErfPuWy8oiAzTgr9RX7Tn7mrLruDXwoOraILyt0GXa8vH5tJtlJKZu5wjA2uc+54fZze+26TK+GkA3hgqlj1n70fpcKGSqkCRuzGssall+Sz7OvIL6WdB4+LWI5OvlJ9zfPTl09GZd/ZR8+tJ+bW1oonztBlfbokOiBZwgE8XYrI5zTdUeECNXeEHfgZOgNjSmBmzKaYGr27lSe9CylGt9YYs8oUhO2HcXACzHzLbGxfGPMqLUnSnW2v9HlfRzBF5DSfwagt2HjDJoyUdIMgsXGShHguAbfui2XFouAkT1TBX4ych2TV9v4/0nalo8YQa28F0lGm1tFNG9h1fZn4ODglAwqXAnCJvMHiLHOTH1MBJl5EstnVovOgwgnaI7kJ3kjUDch99RCRxZ9CpnpZjIKsSDvW0jbP06shoACsPMoP2qGqkf07fnaPAX/7NZMh2LPkdTNx9GC1uSg94xH9E4dyvwzIM3Nd5Vvuq7QY6g3AmH9l92YGugWkWnnMF7qSXxssnQiVyIMSEb5yFUVkJTCxpo09riOR4urjf4szCKCaHT2Uvdhu7Lh1A9hNand29Fm9Ri8Z/1Hp1LB84tLA0pPIaV1ONbFlIcjDI1Xwehya1tVrdKYPy0Ay0Z5OrSg1tDe9xzw24djvZo4kcdfJ1SmC/IlSCNIkextHDTgNuOu3Kdz0dxW0O/J8SL5vggoLZ8VB9a1pG/0Ohei+UXTVGK1sdFXNgnhLoBj2tYa/hHAUUGxkQfYY5uGRhQGMafkmLZzzh389iGDOY8+z23s8hZD2sUyS491ZAvB5lrCWSNF96SqD2HQjwG1IgcLMPGx/vuPuHIFwqofQk8KEY49/SNxVj+gecUkBFjDkFQmiB6IC6AVfguQ2bnZTG6e16u8Kmf3b2JUwIUSsYG9DA/MeEH07tN5Y9j6viRluztr2EgIhl+LsDVBl9reu9MiJWIwO3Dsj/t5b8f+vK/zfiaYn3tIXL74enZjL98xSCnPCfqhDmPYUgR/1DCiHvQZXEppAbjUkzS/nGsqZN/P2fo4Bw5EEllK9RQqvcIv0PRSJSaUhscPt77hW2107LoSNiz1cNR552lfe7MZpr+/3FOL+8GBsj82D3GqTK2gZJN4posUwj8AzhybZ5fnbmfTFGBv57YVlGBoIv8y7G8akzCHaTpHH3T5KlLuDh7EmbSui8xzyvwqd+vm3SfEdELAgycWXM4QnSygFoZhCpOdiT2VYr2+vY1mU9cCv7/QHr81RzzHNtfv57Wx5b6Wm9iSLdf4GB+otxkdRREwVJh7SPOYJYY8ZlSZwWMZg9jCIE7z1wjHOq3QHvE5buW2j4iVElsiW51RImS4KK5YKold5S0cih3mDDagqOCihvo4VFFPM0sCy78OchiH0eoZdMe7c9GFqgg3nWu9AA2/lI2QngkdZsCuBi4wTLm4zrRyMI534Vl4eUZpUFP/AwAY6WlNDiMQk/Tmd+DNCBrkRtquuqCDHHuMyBqNDW+Gn/wPjlZ0z0BlUWRzNMt7SWoYDecYzpHMLGacURWjJLSw7Sd6a+Q188MCVelWC60CQy0dOOvudu3YWTubPD5u2Gq9aAHFy6Bt7KlbOX/gMm4PKAcoiuOPjlZ00ZYNbmBcjcmOyMqc1JJcg8lPFZcScsiPzAxop7YSfhPeYUXTFQWIO6+76h833uQmfnJJqVf8P6ndy8t+dRGAcJRJmFW/BWToFptcYhQvtC6T/88g2dlvucy39uwNcDGM0DCvOgvdwi1iqWOHtmb5LD5WlHgcqyfjh5Of9KWahcjGWqSjS5KvmMk1WN3hJYKsbVet7cX0ZxFILG6+eqfmdGmdQvAoQ1QnrYKzPL/2mZY4G8KcsXxk2xhC3IFAm5qe2pATzPFryxrV7Rxh+61PzYQ54x9yx91n9iJtra0jK2tPVgHF98vDw7/Xwq7FpSWiaE9mPgIe5l4y6DR5ldk5+0K4Ok95NzngdtMCUu3NhfXge+8QBOnlNU12Yu/q3YETaI2oNVF6uJ//e//x2e2UJ79H//5//CZpYq4gOn6B5FRd94xKlcp2ls1uOZEimL6mB7j1EAm4c8D0vOk5t4SRxLJ3KgDKq1y4sR0ot1U1sdrzdK5iEoUVh9HETP+w8mIHp+/art64Gta/Y+zGJ3ZtEug6lmSkNg2d66GRdyBgVDtmc3VXLLSvmdxPqHH344ojYjMqkkGgYITO4vQoPqDAxZZ+DwDidCxuJiVKrN4Lh/ea+QcrAyuKmvKFqlMd0wrsbV0tDTSLo6L/ab7XPECh9IKwOL1dAm0TjaaQBOFSW6z60mwo2SzF1bfbQ1EAGt5gVa6WoJiw2eHhBhYMVhxH5SmUjmgKSiC2sihF2GYNXOpzPLKvi7TElVWy2OmTDLPGglSxLmVFXPoLMit0+hjUfelbZXm6HuK7Cy5Xv3SGu3g+BQ4tjL07Qcqqk+uzg+OpOV1cdHx/90SppCzxE3/cf6PoTZ4LcrVm09MC2almur+4awMZYVuH7XIRbCEiWlf8liklfgC56ho9j3I+vCUVUMSg4lVyY3U7I2JKIc9vtXorWFo7Zv5MPftXXzkTdsDDSngcJrbIgdzaO3rPZw34t0azfdGthAX4dYs1lZ76Syw3UIrJQgQtKTD3FHoYzIy5ceqNzfrNE3adF2qfj/9OqX0xPv5OL8dDVerY399Z2b/X5tf58R9dBJcAc9hT9AxZXuhC5Vy0njXktkpOcyfeVq2A2yhpQEPnuCLG4EaVgQD/A+cmdBhNXbWJQk+w5+PE7zvMpKSntrgqgvZKvaDNo04vj95wK4Gx0MIJ9PSilIUPYIM2tYByv4a8T5QIuTL8s8DE141VQzEbwVDellcyCgj2XDqBtoQ61I6SqsCkoFRIksXSQpYJnuZBU0lYa7eWBF6NxTbdsYY5w/4dwgYgPONmBceAO0bOeoEKMmMu8G0HQOgM9B4Vaq5DB0PlToz4j30xmb1Bzxfy+WFcmYlbttWumwSaFDU+wgoa0qeei+dptn7fKHTUsgGsMSbLLBrpT70I50dxNaJjEk9O9OYwxmBlBsfn+yQVJqFeCmLfKWDBMl+jaSYKuTHF7M6ieLmSPPu5mtczXhMisfeeNckua/Tr/r+QeY77BVsp9Waf+xhjkE12mVYywsWCpHSJ8dlL4REU4MJDI6qY7n/jEnTdm1lDV5q8pkb15MOHCWZtv7QzWDoMq4p7vGd9YCHkzFNYvapA9XLCqdeWt339AucfDr1t1250LLx3v8uVboz3LPkRkSQomNO3rYWitx/K5qxBqMfFp/h7Y4c9UYP3cr+0QcLaOSoiZy2coGDL3hfE4S3tc5HXrXftgDugANSzGebF5/3yTD9KwsLxO/trJklohOL5V9YpuNij7/3HLDhi+7VYbrawuDlDy2FChE6U7DR9/gLkppew0Z6ruLCxtUhC0TOt9XYYhiVUup1ba+f7yqUL7HVQIjE4CRicFoij/hmIAsPOwVqtUy47oimb8RvVLFvtIzkdu8KPHeXEfSDblOrumbR3tCXHBI4Jo3uCYVaDj9AEL9EskapBUYf4agF6WB1AzgCov/ToxfxvGdwpGB3d4XoFq0oxBizKjIZCzYqEyMnwQYIjrm8FwLJQ81BgBY5EA7X3wckF+IqTtrzpm+dCijw1Xb2wC25fCsqJXDiegJ1daJfb8wgKAtNVpXiR682f5rq8bOTOylOjq1bCrtMACpe3GxXeKESVCg+LZS91PJVqAD3rh4+gVWJsGzfGB3l7iyuPt6BrP1c1zi1liMhNVAePcCBFzFs9Uw5BEO+eiV253+Kn3FrYynbvtno9m+MOSzQxi8AkX8pI2knXscFCYZUCCJ/i5I+XTY5Pz9GyP2H8GM1DjxGOvQIXlp8Hn398CnxX0fzj+/G0Tr3Tq0UH0xWibPAU/fcLfh2PxLorad8YSLzJDS4cvmuUSwMyy5ATAkCHqrKpdOybxUlQWdVF3VztoSLK3hQFC64Wil2rJ5cbgNWlYJeEvtlu3lUO5PoePJiUZ0bs1OKyakifsxvWRLnaCMiqK6xs5P9ayEKn/AeyUIJ/FsrT1c1YzZyrowCSd5U6Mqn9Ghoolo/Lkpv2Scpq1Zk5P3014z5UWeVhnMWLtmxJGb/HhpiYkJ8bYDpQ89lt0n7fGnqmAkH5jAg3LDEI7cG5ZgtBS7AgztphPNQZ3KU/Scb3Slz6hfLyEc+bK17FRFOKmJoZoMJEQ75YayZTMRwAiXYSLwYhvcUJ5qZGlV+ME42BhdvVXtmSh8gh47W9Np2wsdYpRLRkl3OBkQF8KhLkEZwET7jK5WCBzjI2bfmVfB5oR0dn6ZQWcsCgfrU4ZO38u8Ov305SN4mcKu8bNFOpdfGXd4gmZe2JJQvXIbLBAE98J9+jYROPvxN70qckh0i8ekvAmx1LtWDUi7Qi3osywsOSImM8C3k0E575dhxZRP99+M5Z0SmJuj3Cayt/K3wxwzYJrP3Doj2Nlaa+bfmno72tEPvOCGmKuWF7fI2k2nGqnxWgXgqFU3OKy+WaJVx65f38DXo6y5N6LhQEK0vWUHYEyJlc1Bg8eH0gtXZCUuNHnu7AKXeZXMOHsk81fNWVldEABw6wC4PEu9P9WvaRhi9tOGyRmp+poGCEqjZbWk+6Yg8BO/YzsRyCPLg9SNTOb2Nm4itgNedygIDlKvII6R5O6XCeqxkdsNluw6JFJv9Qd6Eo23a7EFV3vwExMwnoztlXQdT+3iNsq8AtwXEEs1k3aO8DdSZA0TreSLBZ4xnmhHJVa2TOLYXb+XbiMwuYG+lKdaHWgjUwbvNPvq02a3+dWWSqSlRqcT4LMcEzpeu+D6q53zZRGg6GhNCUWrbyfBLarN9FQf1HtxFpPf6opnm2qQEdWpPfG5Kpk30PnZupmSYYZWHL9O2kiQ1vw6ZoMJRPAs+0Hty2MnW2LaOVD51UYFBaJbZKDuQqtNC+xM9LAlZ1kdmfgGkaKi8RgAaRRX+Uqv/b7zcNoTBY3m3ENbg5cznOsyXr2Ral7mgbS6fIFsJx8rfkHfjKmEL/hTHyblWDDgRTduSaq3X/9Td28NYPWhNewkxvAXsNFzN+/UpZHOp/1y5G+NHCBvgVIKpbKHnpKPYJShfnG48GePHvBmVXgaEOjIqI1W9EdbqJy8b5Z+ZispfL7yQrlYg77ZiDJd2uVfjdpuQDTFiIMViPNs3DvV8E15YeO5vExMv0fs28AtYnx52OAUtHvDOj4g7hNGu7sd8/8Pb123vkeBvAXX1d2F8UCBM2WzhVbdrOVuO8XPDddr3thuxxnrel/qmgds/aqNzXAMohwiPYdo6T53TRWZWOwXZUK4Yz9tb6vIGQX6XlYtSqxxY9TTBtAcfCQhl6mV3UBNyZLQIqRxt42oA2IVsXuU4OM4e+i1yqlh9N0M0RRycjuPGvAsZB5k01JGIVNulP7Cj7VWkBWocn9WtmgSZAJL+0HEvFnsF4V0N8T4KRkvHU8+9OATDuR5/TTRZPx6+lyPtaois10Eefrr0dmXI0x2aZWQsrixVwa56Gwk0ZWBm+3UUEKCM7581ZqpXYJCsOjmQNxI4T0UzJnICODQWA687G6qaAF6tr7yAb4ZP7/ffwuo3OlRFRL0grjxqneaGiDrexCtKpTJ4DbL9JU7fKvKSXgXxmlGFwosqO4dgzWsK0A5QaTq1xDT9UsxJsMRVUbexbcXPJ7UpQKyC2Ed5vzxqg6Z3acHzfpLR4Qr4OoEcUErB7ql1bZpmT4Pn1vfBZJiwYG67k0O+t5LKSkKwPpmVwdr7dKZNdUrp155Onloe4jfAWQJ3BlBGOOlI4DHu5kQGFcFPvpJxgBR4qV5BO18rpvgTtrbIsTT1Uht/Uasp76X89678CgpB9Dx25X6ltviQppk+YwHabsSQkdFbdNpj2zRxkZr0jzsgIzwPs1oHiGmPHVbUOyMOXwJRedWrt2Dl5RsKtCiG+eot4Er/JgvyAVbjzk4SQhHPPfsxeb6Qo3roZTIjXaGq6swzeDzy+ai0FY1Hc+QVpHL8BNgjNnwGfoLtebYMeANP5jxDZ2pp7IaTvLJakaVocVTPl9RmC+aiq2KautOTn/lMzfXIYwaMqUMylYKS10Li5sdunqV2x+IOB2FL2SqIg7CVRvkh03irakZzrTUG/d2FnF6bYrtXcRgd9vbVrkWfdMVd7gzrWIrw6Njyx2hKd5earS9gYtyCIwdbrSR28+zLic9z2HahyknvNPK4rVWXmUvV9yW8P7L2ZniYOJdNn+FY1xGcapYGTgnDztlX+HDLK6CplKVspjFBsnV4cRqMTEhYCXNQfcUUGBEn5qIiL82Gt7qJl+xe+sQTg2yN6rV8hmpo/WPmtqA7y9cpfnUdHxm60VfVLGbLY831uCem9y5ccJ5uSAsZnl0HRoZERsc/ay6pgNhML8Rruljcyss9/USV0u4j9/s1Tn38QF+kXn38bu955raBd293ZkTEaRzJYUtvsD/x8LqRxu4CtyUNnRs3m6yaRNs6PA0doGVshM9Qe8lw/UHTeqlJL+dOYIqCMwioCvjbESAEq+WPZlawwdfYTUBxCs3Gd7p0jx+1kGKu5E07lMRPNNQLjtadrOL9YQfn+2a9eABfMQyBxjuefcpURtg/mKRh6ixA10H8Y3L3VnkeXdXoFDU0cjTQNROppJ91RMIvdwBjaPnImgxRMLZZRmg5J2Qve9ByONZ5G+sdjBYrnL2MPpQ6Qy5ngpolLozp2PMdbKNA/IW0CFgK2P5QdCrI//WQGqvRPcbOVclRAcJkF1MUn+KllVMV/KhoQhy/z6pr/sXa8B9oiTSaEHHeGEpn1sVWY1xY6sIhk1u/WxW2MVHH7kLVewRb3mwhuBbRWHxvacf59rpx+7hx7l++JEOlDT8Otmb4knH7kFH7Zxj01ZF3bQjKSXnNnxEqVxpFrS8mp4q7ZgMiIJ6uv95UDJRQOZRQCJqcVKTUOirLK5onr2S9TQ9q3w9d7knK7D+EKxD20b/et7JmRAxNhD6rozXCUscqVu+BghLQaAwUkod0gXrGMFxHpTy0fV88PmG4Eky/baME0j9wUuwlBRTfEDgVss1AV8j9msk/XmFEPFyeOfv33h3hYcR9B+QIxof4bWXfAPx4m4vixa36yestBlxTFDX8JK3iS6gx1KiJ4G0/U9sxQpFTxI1glwvoyjpskH2fZhxCwrUcCPLqz0C+brtKlGuOvSxhIuxBGVb3uRhOKKaW2XFRdt9+8l9LVeR3CkPY45Gc4tiFnHIN/P8gG6QS5MavCGOzs68n48+Ub3mJ+/y6NOn0xPBexaxuS5wK1ZHbFa7wlM/5vN7AeoxA5+H0yJZq+vuyWJBcfTZOzs9+vTZuzg/rWfpvT/6cHZ64l1ceV/Ofz29+vD+A0y5IZiOQkOx60ePVdGTxHX8954WpZz/MMF54ucXn72r0+OLqxNczQ51ngeZY+ZXMt86ERcg3KNmY9nQtB9WgELwX+YgrgZ0SZfQZIZBIDyh5a7PMWU5aFd1wwbuxVd56GiE/nwDsBgcewbqN3bq6oBR/StCM9zJL8oqeLQ5bwxBRBanj5iB8Ol6TvxNHLBY+gAfwM1AMxLSCcOYiXkTZcYNRMRAZByc0ngYBaYGjAkO8h2eX4y5NOw+zW+zCC9KlKNCtKN+iSjCXMKS77jQxrwK+dd9oKXK04yaci4s/kdtCUhHcYU/XoQUqxJ1L0B9yfEIDwpFJR22C8J0Pi8cMX1W93lc+lmYc6CE5/SCKMCcIt3VXGK+YgyBK2jyJdKQqo6NSCMFMsEoRkbChQ0iLgYEr1EWRuABI7ryBwmC52cjDJNxROzJIxR1UXOWprEqceBfYYI5YNEG9H6UAXR7bDkMwdHjhQ3CMhWSMVNsFpeRdXpaFzHI0nq+iFqpZi3M3RoM8SDkr5aExMTUa8HqihatMoPiCbyZvy4UUwutt/JnYK/BeFNLGdGq0NZTDKy1mvbDSVAahTtJUKMMRS6MiBYANCYM1M9QcPk70hOu24pemboU/PUUdX/LEldrwqSltJOPKe1eeIPTtF7oWwTtnkUJypN7KgNKX37aZ6W514JH3KncUAS8wkH6jO3AT6vF8M+NMwitl/0gaqZHF16lS9rKzAiuEovVmuJGq++x10V/lHTnIZHID4pl6G40Ug8Pip9cF9CQT1r7F/hl2t0wp989wZFwH2LDTNCEIK09mf1FIcsJAmsgFvGqXjQS9OOPofu10WtmPxwd554+6EftQIkXIpVpm8qFtcnAEsj52dn3jNmkCrThVsYJeQQhCvtIXhDNpcH7k6NutUCbBgZeJY/S5Y/j7hlFMOXgUJR4yjB7zFReMm6uQaA9AK0V0PAWLav8rb0QPJikfE8+AQzadIsWtv8AGAMsvAoGuxfmvn0As1nQ+Utzf9/50X7jvLXs2H/E67AFn+8lAyHatbn+g21q5sfG2Ub4YzGqagiGsjc0SLb4m0rgjOTPFNWMNshYL//Z1MKBfVc/59D+Bcl/PQK71tUwoF4G7Py0Z5c3TOx2DOVD1940TE96tKPIaBCltxorCHrLarZesgHTVgTDgxbBnzak/+CEqPCu/dzMCjwVV9iP8MRlXOz5EpgtHQl75mfEi69t+gkdt33RsgRFF2hGZRyaiusO5dNHcKPN0Z6z9xqvHnxNzx99iEno7dL/mubeXImG2ZUV82HpP7j71uCAD4SQKU4fwlklK6HlXoMsXcHBFjleGkl3zDsHWnUhSAMoB4kjA/ql/mVOPlknOV/P8HQA7E+dOFygBaDZu4LOPnFNtbDjdOaK67BVOoCHmwqsRkZigX588WdSx2SN2Qs3/s9/GJ9OutD8uxCPf2k69320gBBif0/zPbJkIewgi9yDg73f1T+Y6zXioLFmcVqEZnNV/sMszErjAyk9sqXjbkn5x1pFUnxAJ5/wp2LHkrXnEE7gdlEaVLMwOETsAA0DS1mzWt2pQ5B4V0FdnKKlYQouitIzMEKma9rlUnKnlX5q1cPoWmU0/cxVP8HqHOWLChf+Er+py7b8zMH6al++M8VsCajwr8OgQiL9DOqFy7vo10VJ6yFl1M+S9WCMRhilCZvOCuL9NrbcLeTLbnBZji8PvBNsZK2EAfRYC+KKN3jWQCBHhkEAoWsI6hdPV/YjjaQRoa1npzUgrDZZBaPRYBqgAfXeAOO88cq50JFoDRZWVNiUiNOB4NOVMORtfQMUWdmFj0P3e7xZ2SPOZXM6v1p3ALW5t4bkeAvuin5r1olrFTj5Mtj7wNlb2RvU8Qg8tZE8Id6f419XkwX05ErSvF1NGkzDDPTYAwcnzlzB2/KY2hxzyoYddvo96NoVx9IUuacMKs1Ar92nKFdte9IPTK9mgiVWd6h6m2GKH+yt7a0V46xYsdX9ZXGKsGV1rShK/LG5Mq9C8cKgmLDo0+71av0hPcKRvDuy33c1Z6EhGcnrtr4LVYgnVjLGu9XKoikE5W7gFzcy3f49Teiu6/mtLfCiVOkkBKAeOCJR4nmywBobua32+jE6UPbQhXX8WPttrs7JbW5GJmDcvndjsCEZifFifSNpQsZ6ddDW/weKQruq'}
HASHES={'mpkg_v1.json': '1cf3099355bf17cf5637445907ea4b37d36ffb6301726c671dfbc0cbe2ae095f', 'm4_build_dataset.py': '380004fc21ad7985054991e7f578846ebaf7dbac25812517e3bdfa0a2446c098', 'm4_runner.py': '75a00be3aaf41c359b23e25c31bbcfcd1be1fc4edc4093e36914887f2fc2e2b5'}
for name, b64 in PAYLOAD.items():
    data=zlib.decompress(base64.b64decode(b64))
    assert hashlib.sha256(data).hexdigest()==HASHES[name],f'corrupt embedded file: {name}'
    path=WORK/name
    if path.exists() and path.read_bytes()!=data:raise RuntimeError(f'Existing source differs: {name}')
    if not path.exists():path.write_bytes(data)
print('PASS: authenticated embedded source; script hash:',HASHES['m4_runner.py'][:16])


## 2 — Deterministically regenerate and verify all 40 + 240 + 80 facts (CPU)

In [ ]:
DATA=WORK/'M4_CP2_Data'
subprocess.run([sys.executable,str(WORK/'m4_build_dataset.py'),'--source',str(WORK/'mpkg_v1.json'),'--output',str(DATA)],check=True)
manifest=json.loads((DATA/'manifest.json').read_text(encoding='utf-8'))
for name,digest in manifest['record_hashes'].items():
    assert hashlib.sha256((DATA/name).read_bytes()).hexdigest()==digest,(name,'hash mismatch')
print('PASS: source-backed 40 obsolete, 240 retained, 80 held-out workpiece facts')


## 3 — Install tested dependency pins and mount Drive (GPU)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3',
    'peft==0.15.2','accelerate==1.6.0','bitsandbytes==0.45.5','safetensors>=0.5.0',
    'sentencepiece>=0.2.0'],check=True)
import torch, shutil
if not torch.cuda.is_available():raise RuntimeError('GPU required. Runtime > Change runtime type > T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2),'GiB')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/M4_Manufacturing_Research')
except ImportError:
    raise RuntimeError('Use Google Colab for the GPU stage and Google Drive checkpoint persistence')
# Never overwrite the v1 pilot or another experimental version.
PROJECT=BASE/'M4_v2_Accelerated_Full_Study'
PROJECT.mkdir(parents=True,exist_ok=True)
for name in ('mpkg_v1.json','m4_build_dataset.py','m4_runner.py'):
    dest=PROJECT/name; src=WORK/name
    if dest.exists() and hashlib.sha256(dest.read_bytes()).hexdigest()!=HASHES[name]:
        raise RuntimeError(f'Existing v2 experiment code differs: {dest}. Do not mutate a frozen experiment')
    if not dest.exists():shutil.copy2(src,dest)
DATA_DRIVE=PROJECT/'M4_CP2_Data';DATA_DRIVE.mkdir(exist_ok=True)
for src in DATA.iterdir():
    dest=DATA_DRIVE/src.name
    if dest.exists() and dest.read_bytes()!=src.read_bytes():raise RuntimeError('Dataset changed: '+src.name)
    if not dest.exists():shutil.copy2(src,dest)
from huggingface_hub import HfApi
rev_file=PROJECT/'MODEL_REVISION_SHA.txt'
if rev_file.exists():REV=rev_file.read_text().strip()
else:
    REV=HfApi().model_info('Qwen/Qwen2.5-0.5B-Instruct').sha
    rev_file.write_text(REV+'\n',encoding='utf-8')
assert len(REV)==40
os.environ['M4_MODEL_REVISION']=REV
os.environ['M4_LOCAL_MODEL_CACHE']=str(WORK/'model_cache')
(PROJECT/'ENVIRONMENT.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
print('Checkpoint directory:',PROJECT,'\nQwen revision:',REV)


## 4 — Freeze v2 development training plan and runtime helpers

In [ ]:
# These are locked for v2. Do not modify after final test begins.
SEEDS=[20261101,20261102,20261103]
EPOCHS_OR=5
LR_OR=2e-4
OBSOLETE_REPEAT=6
EPOCHS_UC=3
LR_UC=1e-4
BATCH=8
GEN_BATCH=8
MAX_GENERATION=96
RUNNER=str(PROJECT/'m4_runner.py')
COMMON=['--data',str(DATA_DRIVE),'--out',str(PROJECT)]
import datetime, traceback
RUN_START=time.monotonic()
def run_stage(command,*args):
    commandline=[sys.executable,RUNNER,command,*COMMON,*map(str,args)]
    label=' '.join(map(str,args))
    print('\n>>',datetime.datetime.now().isoformat(timespec='seconds'),command,label,flush=True)
    subprocess.run(commandline,check=True,env=os.environ.copy())
    print('<< elapsed runtime min:',round((time.monotonic()-RUN_START)/60,1),flush=True)
def train_original_and_reference(sd):
    for role in ('O','R'):
        run_stage('train','--role',role,'--seed',sd,'--epochs',EPOCHS_OR,'--lr',LR_OR,
                  '--batch',BATCH,'--obsolete-repeat',OBSOLETE_REPEAT)
def gate_original_and_reference(sd):
    for role in ('O','R'):
        run_stage('eval','--seed',sd,'--split','dev','--role',role,
                  '--precision','bf16','--gen-batch',GEN_BATCH,'--max-new-tokens',MAX_GENERATION)
    # The gate records the unchanged admission decision. It does not remove failed seeds.
    run_stage('gate','--seed',sd)
def train_unlearned_and_control(sd):
    for role in ('U','C'):
        run_stage('train','--role',role,'--seed',sd,'--epochs',EPOCHS_UC,'--lr',LR_UC,
                  '--batch',BATCH,'--obsolete-repeat',OBSOLETE_REPEAT)


## 5 — Seed 1 O/R development: assess whether greater original learning exposure worked

In [ ]:
train_original_and_reference(SEEDS[0])
gate_original_and_reference(SEEDS[0])
GATE1=json.loads((PROJECT/'evaluations'/str(SEEDS[0])/'development_gate.json').read_text())
print('\nFIRST-SEED GATE:',json.dumps(GATE1,indent=2))
if not GATE1['identified']:
    print('WARNING: unchanged learnability gate FAILED. Continuing *all* measurements for negative-result completeness; no claim of successful unlearning can follow from this seed.')


## 6 — Seed 1 U/C training and limited development checks (full development for U/C BF16)

In [ ]:
train_unlearned_and_control(SEEDS[0])
for role in ('U','C'):
    run_stage('eval','--seed',SEEDS[0],'--split','dev','--role',role,
              '--precision','bf16','--gen-batch',GEN_BATCH,'--max-new-tokens',MAX_GENERATION)
# Actual packed NF4 smoke test, not a final-test sample. Here a small limit only verifies the backend.
run_stage('eval','--seed',SEEDS[0],'--split','dev','--role','U',
          '--precision','nf4','--limit',3,'--gen-batch',3,'--max-new-tokens',MAX_GENERATION)


## 7 — Lock the exact protocol **before accessing any final-test question**

In [ ]:
# Zero test evaluation before this freeze. All final-test data remains in the fixed manifest.
FROZEN={
    'version':'M4_v2_accelerated_20261101','seeds':SEEDS,
    'O_R_epochs':EPOCHS_OR,'O_R_learning_rate':LR_OR,'obsolete_repeat':OBSOLETE_REPEAT,
    'U_C_epochs':EPOCHS_UC,'U_C_learning_rate':LR_UC,
    'batch_size':BATCH,'generation_batch':GEN_BATCH,'max_new_tokens':MAX_GENERATION,
    'dataset_manifest_sha256':hashlib.sha256((DATA_DRIVE/'manifest.json').read_bytes()).hexdigest(),
    'runner_sha256':hashlib.sha256((PROJECT/'m4_runner.py').read_bytes()).hexdigest(),
    'model_revision':REV,'development_gate_seed1':GATE1,
    'principle':'All outcomes including failed baselines retained; no post-test filtering.'}
freezefile=PROJECT/'FINAL_TEST_FREEZE.json'
if freezefile.exists():
    assert json.loads(freezefile.read_text(encoding='utf-8'))==FROZEN,'Final test freeze conflict'
else:freezefile.write_text(json.dumps(FROZEN,ensure_ascii=False,indent=2),encoding='utf-8')
print('Frozen:',freezefile)


## 8 — Finish O/R/U/C training and O/R development for seeds 2 and 3

In [ ]:
for seed in SEEDS[1:]:
    train_original_and_reference(seed)
    gate_original_and_reference(seed)
    train_unlearned_and_control(seed)
print('ALL THREE SEEDS TRAINED. Development gate statuses:')
for sd in SEEDS:
    g=json.loads((PROJECT/'evaluations'/str(sd)/'development_gate.json').read_text())
    print(sd,'identified',g['identified'],'O',g['F_O_dev'],'R',g['F_R_dev'])


## 9 — All 36 full final-test evaluations: three seeds × four models × three representations

In [ ]:
if json.loads(freezefile.read_text())!=FROZEN:
    raise RuntimeError('Final test configuration changed: refusing to proceed')
for sd in SEEDS:
    for role in ('O','R','U','C'):
        for precision in ('bf16','int8','nf4'):
            run_stage('eval','--seed',sd,'--split','test','--role',role,'--precision',precision,
                      '--limit',0,'--gen-batch',GEN_BATCH,'--max-new-tokens',MAX_GENERATION)
print('COMPLETE: all 36 full model/precision final evaluations')


## 10 — Verified reporting from observed outputs only

In [ ]:
run_stage('report','--min-seeds',3)
status=json.loads((PROJECT/'reports'/'M4_status.json').read_text())
assert status['full_group_count']==3*4*3*3
assert status['full_test_record_count']==3*4*3*(40+240+80)
print('\nREPORT CREATED:',PROJECT/'reports')
print(json.dumps(status,indent=2,ensure_ascii=False))
from IPython.display import display,Image
figure=PROJECT/'reports'/'M4_Figure10_candidate.png'
if figure.exists():display(Image(filename=str(figure)))


## Reading the outcomes correctly
- **If any O/R development gate fails**, the complete result is still retained, but it is **not evidence of quantization durability of successfully unlearned knowledge** for that seed.
- **Three seeds are not three manufacturing factories**; application evidence is from a source-grounded simulated revision benchmark, not production validation.
- The test records are untouched until `FINAL_TEST_FREEZE.json` is written. If the protocol needs adjustment, do that as a separately versioned future study; never tune v2 on its test results.
- Results in `reports/` are candidate tables and graphics, not yet independently confirmed journal evidence. Inspect actual answers and anomaly cases before revising the manuscript.
- If Colab disconnects, rerun this notebook from the top: marker-verified training and completed evaluations are skipped; partial `.partial.jsonl` files resume.
- The 4-hour goal depends on available GPU speed, dataset answer lengths, library loading and Colab stability. The notebook will not silently truncate full evaluation to meet a deadline.
